# CNN(VGG16) with PyTorch for FashionMNIST

In [ ]:
import torch
from torch.utils.data import DataLoader
import torch.nn as nn

from torchvision import datasets, transforms
from torchvision.transforms import InterpolationMode
import torchvision.models as models
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score

In [ ]:
# Check the ability of GPU device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
torch.manual_seed(42)

In [ ]:
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3), # 1D to 3D
    transforms.Resize(256, interpolation=InterpolationMode.BILINEAR),
    transforms.CenterCrop(224),
    transforms.ToTensor(), # Convert to float32 and scale to [0, 1]
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_data = datasets.FashionMNIST(
    root="../../datasets/",
    train=True,
    download=True,
    transform=transform,
)

test_data = datasets.FashionMNIST(
    root="../../datasets/",
    train=False,
    download=True,
    transform=transform
)


train_loader = DataLoader(train_data, batch_size=64, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False, pin_memory=True)

In [ ]:
feature, target = train_data[0]
feature.shape

In [ ]:
# Fetch Pretrained model
model = models.vgg16(pretrained=True)
model

In [ ]:
model.features

In [ ]:
model.classifier

In [ ]:
# Freeze CNN part by disabling gradient calculation(tracking)
for param in model.features.parameters():
    param.requires_grad = False

In [ ]:
model.classifier = nn.Sequential(
    nn.Linear(25088, 1024),
    nn.ReLU(),
    nn.Dropout(0.35),
    
    nn.Linear(1024, 512),
    nn.ReLU(),
    nn.Dropout(0.35),
    
    nn.Linear(512, 10)
)
model.to(device=device)

In [ ]:
learning_rate = 0.001
epochs = 10
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=learning_rate)
    # Only update classifier parameters


In [ ]:
# Training loop
for epoch in range(epochs):
    epoch_loss = 0
    for batch_features, batch_labels in train_loader                        :
        
        # Move batch_features and batch_labels to GPU
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
        
        # Forward pass
        outputs = model(batch_features)
        
        # Calculate Loss
        loss = criterion(outputs, batch_labels)
        epoch_loss += loss.item()
        # Clear gradients
        optimizer.zero_grad()
        
        # Backward Pass
        loss.backward()
        
        # Update Gradients
        optimizer.step()
        
    # len(train_loader) returns batch_count
    print(f"Epoch :{epoch + 1}, Loss: {epoch_loss/len(train_loader)}")

In [ ]:
# Evaluation over the full test set in batches
model.eval()

all_preds = []
all_labels = []

with torch.inference_mode():
    for test_features, test_labels in test_loader:
        test_features, test_labels = test_features.to(device), test_labels.to(device)
        logits = model(test_features)
        preds = logits.argmax(dim=1)
        
        all_preds.append(preds.cpu())
        all_labels.append(test_labels.cpu())

preds = torch.cat(all_preds).numpy()
labels = torch.cat(all_labels).numpy()

accuracy = accuracy_score(labels, preds)
precision = precision_score(labels, preds, average='macro', zero_division=0)
recall = recall_score(labels, preds, average='macro', zero_division=0)
f1 = f1_score(labels, preds, average='macro', zero_division=0)

print(f"Accuracy: {accuracy}, Precision: {precision}, Recall: {recall}, F1 Score: {f1}")